# CYNIC Mistral 7B Fine-Tuning — Phase 2

**Runtime**: Google Colab (Free T4 GPU, 16GB VRAM)

**Duration**: ~1.5 hours for 3 epochs

**Goal**: Fine-tune Mistral 7B for governance judgment using Unsloth QLoRA

## Instructions

1. Open in Colab: [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/your-repo/blob/main/CYNIC_Mistral_Finetune_Colab.ipynb)

2. Ensure GPU is selected:
   - Runtime → Change runtime type → Hardware accelerator: **T4 GPU**

3. Run cells in order (top to bottom)

4. Download LoRA adapters when complete

---


## Cell 1: Check GPU and Install Dependencies

In [ ]:
# Check GPU
import torch
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected. Change runtime to T4 GPU.")

In [ ]:
# Install Unsloth and dependencies
# This takes ~2 minutes
import subprocess
import sys

print("Installing Unsloth + dependencies...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "unsloth[colab-new]"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "transformers", "datasets", "trl", "accelerate"])
print("✓ Installation complete!")

## Cell 2: Upload Training Data

Either upload the JSONL file or load from local path.

**Option A**: Upload `governance_v1.jsonl` from your machine (recommended)

**Option B**: Generate it here (if you don't have it)

In [ ]:
# OPTION A: Upload training data
# Uncomment below and upload your governance_v1.jsonl file

from google.colab import files
import os
from pathlib import Path

print("Uploading training data...")
uploaded = files.upload()

# Verify upload
if 'governance_v1.jsonl' in uploaded:
    print("✓ File uploaded successfully!")
    training_file = Path('/content/governance_v1.jsonl')
else:
    print("No file found. Try uploading governance_v1.jsonl")

In [ ]:
# OPTION B: Generate training data here (if you don't have it)
# Uncomment this section to generate 14 synthetic examples

import json
from pathlib import Path

# Define synthetic proposals
SYNTHETIC_PROPOSALS = [
    {
        "title": "Allocate 5% treasury to community marketing budget",
        "description": "Use 50,000 tokens from treasury to fund community-run marketing campaigns. Community votes on spend. Unused funds burned.",
        "category": "BUDGET_ALLOCATION",
        "impact_level": "MEDIUM",
        "expected_verdict": "HOWL",
        "expected_q_score": 60.0,
    },
    {
        "title": "Implement E-Score reputation system",
        "description": "Track proposer/voter reputation. Public scores. Prevents Sybil attacks.",
        "category": "GOVERNANCE_CHANGE",
        "impact_level": "HIGH",
        "expected_verdict": "HOWL",
        "expected_q_score": 59.0,
    },
    {
        "title": "Pay core team $5K/month",
        "description": "Hire 2 community managers. Can be fired by community vote. Monthly reports required.",
        "category": "BUDGET_ALLOCATION",
        "impact_level": "MEDIUM",
        "expected_verdict": "WAG",
        "expected_q_score": 50.0,
    },
    {
        "title": "Allocate 20% treasury to founder venture fund",
        "description": "Give founder $50K from treasury. Founder keeps all returns. No oversight.",
        "category": "EXTRACTION",
        "impact_level": "CRITICAL",
        "expected_verdict": "GROWL",
        "expected_q_score": 30.0,
    },
    {
        "title": "Transfer 50% treasury to founder wallet",
        "description": "Move funds to founder for 'safekeeping'. No smart contract protection.",
        "category": "EXTRACTION",
        "impact_level": "CRITICAL",
        "expected_verdict": "BARK",
        "expected_q_score": 15.0,
    },
]

SYSTEM_PROMPT = """You are CYNIC, a governance intelligence organism. You judge governance proposals using 5 axioms:

AXIOMS (in order of importance):
1. FIDELITY — Does this proposal faithfully represent community intent? (70%)
2. PHI — Is the reasoning φ-bounded (0-61.8%) or runaway? (10%)
3. VERIFY — Can the proposal be audited and enforced? (10%)
4. CULTURE — Does it strengthen or weaken community governance? (5%)
5. BURN — Are community funds burned (not extracted) or do founders profit? (5%)

VERDICT RULES:
- HOWL (Q ≥ 61.8): Strong proposal. Non-extractive, clear, auditable.
- WAG (Q 38.2-61.8): Good proposal with minor concerns.
- GROWL (Q 23.6-38.2): Risky proposal. Extraction signals.
- BARK (Q < 23.6): Dangerous proposal. Rug risk, founder extraction.

RESPONSE FORMAT: Return valid JSON with verdict, q_score, confidence, reasoning."""

# Generate examples
training_file = Path('/content/governance_v1.jsonl')

with open(training_file, 'w') as f:
    for prop in SYNTHETIC_PROPOSALS:
        proposal_json = json.dumps({
            "title": prop["title"],
            "description": prop["description"],
            "category": prop["category"],
            "impact_level": prop["impact_level"],
        })
        
        verdict_json = json.dumps({
            "verdict": prop["expected_verdict"],
            "q_score": prop["expected_q_score"],
            "confidence": 0.95,
            "reasoning": f"Training example for {prop['expected_verdict']}",
        })
        
        example = {
            "messages": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": f"Judge this governance proposal:\n\n{proposal_json}"},
                {"role": "assistant", "content": verdict_json},
            ]
        }
        f.write(json.dumps(example) + "\n")

print(f"✓ Generated {len(SYNTHETIC_PROPOSALS)} training examples")
print(f"✓ Saved to {training_file}")

## Cell 3: Verify Training Data

In [ ]:
# Verify training data
import json
from pathlib import Path

training_file = Path('/content/governance_v1.jsonl')

if not training_file.exists():
    print(f"ERROR: Training file not found: {training_file}")
else:
    # Count and categorize
    verdicts = {}
    count = 0
    
    with open(training_file) as f:
        for line in f:
            if not line.strip():
                continue
            example = json.loads(line)
            verdict = json.loads(example["messages"][2]["content"]).get("verdict")
            verdicts[verdict] = verdicts.get(verdict, 0) + 1
            count += 1
    
    print(f"Training Data Verification")
    print(f"=" * 50)
    print(f"File: {training_file}")
    print(f"Total examples: {count}")
    print(f"\nVerdict distribution:")
    for v, c in sorted(verdicts.items()):
        print(f"  {v}: {c}")
    print(f"\n✓ Ready for training!")

## Cell 4: Load Model and Tokenizer

In [ ]:
from unsloth import FastLanguageModel
import torch

print("Loading Mistral 7B (4-bit quantized)...")
print("(This may take 1-2 minutes)")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
    max_seq_length=2048,
    dtype=None,  # Auto-detect
    load_in_4bit=True,
)

print("Adding LoRA adapters (rank=16)...")
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

print("✓ Model loaded!")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6:.1f}M")

## Cell 5: Prepare Dataset

In [ ]:
import json
from datasets import Dataset
from pathlib import Path

training_file = Path('/content/governance_v1.jsonl')

print("Loading training data...")

# Load JSONL
raw_data = []
with open(training_file) as f:
    for line in f:
        if line.strip():
            raw_data.append(json.loads(line))

print(f"Loaded {len(raw_data)} examples")

# Format messages for training
def format_messages(example):
    messages = example.get("messages", [])
    text = ""
    
    for msg in messages:
        role = msg.get("role", "")
        content = msg.get("content", "")
        
        if role == "system":
            text += f"<s>[INST] <<SYS>>\n{content}\n<</SYS>>\n\n"
        elif role == "user":
            text += f"{content} [/INST] "
        elif role == "assistant":
            text += f"{content}</s>"
    
    return {"text": text}

# Apply formatting
formatted_data = [format_messages(ex) for ex in raw_data]

# Create dataset
dataset = Dataset.from_list(formatted_data)

# Tokenize
def tokenize_func(example):
    return tokenizer(
        example["text"],
        max_length=2048,
        truncation=True,
        padding="max_length",
        return_tensors=None,
    )

print("Tokenizing dataset...")
tokenized = dataset.map(
    tokenize_func,
    batched=True,
    remove_columns=["text"],
    desc="Tokenizing",
)

print(f"✓ Prepared {len(tokenized)} tokenized examples")

## Cell 6: Fine-Tune Model

**This will take ~1.5 hours for 3 epochs.**

Training parameters:
- Batch size: 2 (per device)
- Gradient accumulation: 4 (effective batch = 8)
- Learning rate: 2e-4
- Epochs: 3
- Save strategy: save after each epoch

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer
import os

output_dir = "/content/cynic-mistral-7b-qlora"
os.makedirs(output_dir, exist_ok=True)

print("Setting up training arguments...")
training_args = TrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=10,
    num_train_epochs=3,
    learning_rate=2e-4,
    fp16=not torch.cuda.is_available(),
    bf16=torch.cuda.is_available(),
    logging_steps=5,
    optim="paged_adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    seed=42,
    save_strategy="epoch",
    save_total_limit=2,
)

print("Creating SFT Trainer...")
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=tokenized,
    dataset_text_field="text",
    args=training_args,
    packing=False,
    max_seq_length=2048,
)

print("\n" + "="*80)
print("STARTING TRAINING (3 epochs)")
print("="*80 + "\n")

trainer.train()

print("\n" + "="*80)
print("TRAINING COMPLETE!")
print("="*80)
print(f"\nLoRA adapters saved to: {output_dir}")
print(f"Ready for download!")

## Cell 7: Verify Training Results

In [ ]:
import os
from pathlib import Path

output_dir = Path("/content/cynic-mistral-7b-qlora")

print("Training Results Verification")
print("="*50)

# Check files
files = list(output_dir.glob("**/*"))
print(f"\nLoRA adapter files:")
for f in sorted(files):
    if f.is_file():
        size_mb = f.stat().st_size / 1e6
        print(f"  {f.name}: {size_mb:.1f} MB")

# Total size
total_size = sum(f.stat().st_size for f in files if f.is_file()) / 1e6
print(f"\nTotal LoRA size: {total_size:.1f} MB")

print("\n✓ Ready to download!")

## Cell 8: Download LoRA Adapters

Download the trained LoRA adapters to your local machine.
They will be used in Phase 3 (export to Ollama).

In [ ]:
import shutil
from pathlib import Path
from google.colab import files

output_dir = Path("/content/cynic-mistral-7b-qlora")

print("Preparing download...")
print("This will create a ZIP file with all LoRA adapters")

# Create zip
zip_path = Path("/content/cynic-mistral-7b-qlora.zip")
shutil.make_archive(str(zip_path.with_suffix('')), 'zip', str(output_dir.parent), output_dir.name)

print(f"\nZIP file created: {zip_path}")
print(f"Size: {zip_path.stat().st_size / 1e6:.1f} MB")

print("\nDownloading...")
files.download(str(zip_path))

print("\n✓ Download complete!")
print("\nNext steps:")
print("1. Extract ZIP on your machine")
print("2. Update path in Phase 3 export script")
print("3. Run: python -m cynic.training.export_ollama")

## Summary

✓ **Phase 2 Complete!**

**What happened:**
- Loaded Mistral 7B with 4-bit quantization
- Added LoRA adapters (rank=16, 48MB total)
- Trained for 3 epochs on governance proposals
- Downloaded adapters to your machine

**Next: Phase 3 — Export to Ollama**

```bash
# On your local machine
python -m cynic.training.export_ollama \
  --model-dir /path/to/downloaded/cynic-mistral-7b-qlora
```

This will:
1. Merge LoRA adapters with base model
2. Create Ollama Modelfile
3. Register as `cynic-mistral:7b`
4. Ready for benchmarking

---

**Questions?** See `MISTRAL_FINETUNE_PLAN.md` in the repo.